# MAAI801 — Max-CSP Machine Learning Mini-Project
**Student starter notebook | Teams of 4**

**Your work:** Explore the dataset, train and compare Decision Tree/Random Forest regressors, then compare ML-guided variable selection with the template heuristic.

**Provided:** Synthetic data, constraint evaluation, best-value modification, and baseline/ML evaluation functions. These are **not** the full genetic algorithm described in the research paper.

### First-time setup
- **Local Jupyter:** Keep this notebook with the project data files, then run the notebook in order.
- **Google Colab:** Upload `student_starter.ipynb` using *File → Upload notebook*. In the left **Files** pane, upload **`MAAI801_MaxCSP_Student_Package.zip`** (the ZIP supplied by your instructor). Then run the setup cell immediately below: it will locate and extract the provided files.
- The model-training sections are intentionally incomplete. Complete all `TODO` tasks. Do **not** use the `improvement` target as an input, and do **not** use the test split while choosing hyperparameters.


In [ ]:
# Setup — works with files beside the notebook OR with the uploaded student ZIP in Colab.
from pathlib import Path
import sys
import zipfile

cwd = Path.cwd()
candidates = [cwd] + list(cwd.glob('*/'))
ROOT = next((p for p in candidates if (p / 'candidate_dataset.csv').exists()), None)

if ROOT is None:
    for archive in sorted(cwd.glob('*.zip')):
        with zipfile.ZipFile(archive) as z:
            if any(n.endswith('candidate_dataset.csv') for n in z.namelist()):
                z.extractall(cwd)
                break
    candidates = [cwd] + [p.parent for p in cwd.rglob('candidate_dataset.csv')]
    ROOT = next((p for p in candidates if (p / 'candidate_dataset.csv').exists()), None)

if ROOT is None:
    raise FileNotFoundError(
        'Missing candidate_dataset.csv. Upload the provided student ZIP in the Colab Files panel, '
        'or place all project files in the current working folder.'
    )
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))
print('Project files found in:', ROOT)
print('Required dataset and Python tools:', all((ROOT / x).exists() for x in
      ['candidate_dataset.csv', 'states.csv', 'instances.json', 'maxcsp_tools.py']))


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from maxcsp_tools import load_instances, candidate_features, evaluate_heuristic, evaluate_ml_selection

# ROOT is initialized by the setup cell above.
data = pd.read_csv(ROOT / 'candidate_dataset.csv')
states = pd.read_csv(ROOT / 'states.csv')
instances = load_instances(ROOT / 'instances.json')
FEATURES = ['variable_degree', 'variable_violations', 'template_percentage',
            'current_violations', 'current_satisfaction', 'num_constraints']
TARGET = 'improvement'
print('Dataset shape:', data.shape)
print('Number of instances:', data.instance_id.nunique())
print(data.groupby('split').size())
print(data.head().to_string(index=False))


## Provided baseline demonstration — run before the ML tasks

This demonstration selects one test-independent **training state**, applies the percentage-based heuristic, and evaluates the selected variable. It is provided so you can understand the baseline. You must still build **both regression models** and complete the ML comparison yourselves.


In [ ]:
# Provided baseline: one starting solution and one best-value modification.
s = states[states['split'] == 'train'].iloc[0]
inst = instances[s['instance_id']]
assignment = json.loads(s['assignment_json'])
from maxcsp_tools import n_violations, satisfaction, heuristic_select
print('Starting assignment:', assignment)
print('Before: violated =', n_violations(inst, assignment),
      ', satisfied =', satisfaction(inst, assignment))
selected = heuristic_select(inst, assignment)
result = evaluate_heuristic(inst, assignment)
print('Selected variable index:', selected, '(Python indices start at 0)')
print('Change:', result['old_value'], '→', result['new_value'])
print('After: violated =', result['violations_after'],
      ', satisfied =', result['satisfied_after'])
print('Actual one-step improvement:', result['improvement'])


## Task 1 — Explore the data
Explain: What is one row? What does the target measure? Why must we split by `instance_id` rather than random rows?
Create **at least two visualizations**, and interpret them.


In [ ]:
# TODO (Student 1): Add at least two useful visualizations and explanatory comments.
print(data[FEATURES + [TARGET]].describe())


## Task 2 — Train a Decision Tree Regressor
Fit on the provided `train` instances. Evaluate on a group-wise validation subset of train instances first; reserve `test` for the final evaluation.


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit

train_df = data[data['split'] == 'train'].copy()
test_df = data[data['split'] == 'test'].copy()
assert set(train_df.instance_id).isdisjoint(set(test_df.instance_id))

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_tr, idx_val = next(splitter.split(train_df, groups=train_df['instance_id']))
fit_df = train_df.iloc[idx_tr]
val_df = train_df.iloc[idx_val]

# TODO (Student 2): Train a DecisionTreeRegressor and report validation MAE.
# Example: dt_model = DecisionTreeRegressor(max_depth=..., random_state=42)
# dt_model.fit(fit_df[FEATURES], fit_df[TARGET])


## Task 3 — Train a Random Forest Regressor
Train, tune, and evaluate a random forest using the **same** fit/validation instance split. Compare with the Decision Tree.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# TODO (Student 3): Define and fit rf_model; evaluate its validation MAE.
# Compare at least two hyperparameter settings without examining the test set.


## Task 4 — Choose a model and evaluate on the held-out test set
After selecting your model using validation performance, evaluate MAE on held-out `test` instances **once**. Then compare ML-selected vs heuristic-selected variables on the exact same search states.


In [ ]:
# TODO (Student 4): Assign your final fitted model to best_model.
# best_model = ...
# print('Test MAE:', mean_absolute_error(test_df[TARGET], best_model.predict(test_df[FEATURES])))

# Evaluation example — uncomment after best_model is fitted.
# rows = []
# for s in states[states['split'] == 'test'].itertuples(index=False):
#     inst = instances[s.instance_id]
#     assignment = json.loads(s.assignment_json)
#     heuristic = evaluate_heuristic(inst, assignment)
#     ml = evaluate_ml_selection(inst, assignment, best_model, FEATURES)
#     rows.append({'state_id': s.state_id,
#                  'heuristic_improvement': heuristic['improvement'],
#                  'ml_improvement': ml['improvement'],
#                  'heuristic_variable': heuristic['selected_variable'],
#                  'ml_variable': ml['selected_variable']})
# comparison = pd.DataFrame(rows)
# display(comparison.head())
# print(comparison[['heuristic_improvement','ml_improvement']].mean())


## Task 5 — Reflection (answer in notebook)
1. Which model achieved lower validation and final test MAE?
2. How often did ML select a different variable from the template heuristic?
3. On which instances did ML improve, match, or worsen the baseline's **actual** one-step improvement?
4. Could a low prediction error still produce poor variable choices? Explain.
5. Do the results justify claiming a faster or better **complete** genetic algorithm? Why or why not?
6. What is one limitation of synthetic instances or this simplified mutation procedure?
